# Pilot batch dashboard (batch deep-dive)

Reads the same read-only data layer as `live/monitor.py` and `live/web.py`
(`live/reader.py`) for on-demand plots not needed in real time: t_gate
histograms, full dominant-share distributions, per-topology eligibility, and
cost summaries (WORKING.md §9).

By default this notebook runs a small **mocked** pilot batch into a temp
directory so it executes cleanly offline (`--model mock`, per the §0
zero-blocker build contract). Point `DATA_DIR`/`LOGS_DIR` at a real
`data/`/`logs/` tree to analyze an actual pilot/main-study batch instead.

In [1]:
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("Agg")  # headless-safe (CI / nbconvert)
import matplotlib.pyplot as plt
import pandas as pd

from convention_topology.config import derive_seeds, load_config, resolve_model
from live.cost import cost_summary
from live.reader import load_dashboard_state
from run_pilot import GridCell, run_batch

USE_MOCK_SCRATCH_BATCH = True  # False -> point at a real data/logs tree below


In [2]:
if USE_MOCK_SCRATCH_BATCH:
    _scratch = tempfile.TemporaryDirectory()
    DATA_DIR = Path(_scratch.name) / "data"
    LOGS_DIR = Path(_scratch.name) / "logs"

    config = load_config()
    config.params.N = 8  # keep the mocked demo batch fast
    model_spec = resolve_model(config, "mock")
    cells_grid = [
        GridCell(topology, i, ((i - 1) % 6) + 1, *derive_seeds(1, topology, ((i - 1) % 6) + 1, i))
        for topology in ("complete", "random_regular", "lattice")
        for i in range(1, 3)  # 2 seeds/topology -- fast demo, not the real 12
    ]
    run_batch(
        config, model_spec, cells_grid, data_dir=DATA_DIR, logs_dir=LOGS_DIR,
        mock_policy="imitate_modal", workers=1, retry_failed=False,
    )
else:
    DATA_DIR = REPO_ROOT / "data"
    LOGS_DIR = REPO_ROOT / "logs"

state = load_dashboard_state(LOGS_DIR, DATA_DIR)
print(f"{len(state.runs)} run(s), {len(state.events)} event(s) loaded")
state.runs.head()


6 run(s), 11040 event(s) loaded


,adjacency_hash,cm_k,code_commit,committed_to,config_hash,eligibility,endpoint_base_url,finished_at,graph_seed,incumbent_label,...,n_retries,phase2_horizon,prompt_template_version,run_id,run_seed,run_status,started_at,t_gate,t_phase1_end,topology
0,f40de6943c667698,1,pilot,Q,743a7783a67526f0,1,,2026-07-14T08:03:27.504474+00:00,1256887330,M,...,0,1000,v1-ashery-chat-split,mock_complete_k1_s3040717132,3040717132,completed,2026-07-14T08:03:27.462705+00:00,36,36,complete
1,f40de6943c667698,2,pilot,Q,1130341c81d16392,1,,2026-07-14T08:03:27.541829+00:00,2903169859,M,...,0,1000,v1-ashery-chat-split,mock_complete_k2_s1360069322,1360069322,completed,2026-07-14T08:03:27.504997+00:00,44,44,complete
2,de57487ee1c345a3,1,pilot,M,075849ed0fee699a,1,,2026-07-14T08:03:27.666262+00:00,3298650775,Q,...,0,1000,v1-ashery-chat-split,mock_lattice_k1_s2323611162,2323611162,completed,2026-07-14T08:03:27.624472+00:00,57,57,lattice
3,de57487ee1c345a3,2,pilot,Q,f8bc3010f8f27877,1,,2026-07-14T08:03:27.705087+00:00,1237715406,M,...,0,1000,v1-ashery-chat-split,mock_lattice_k2_s3017999296,3017999296,completed,2026-07-14T08:03:27.666779+00:00,67,67,lattice
4,e46fd8f9bab950ad,1,pilot,M,a3c7c0b7eff6cfeb,1,,2026-07-14T08:03:27.586148+00:00,2340861474,Q,...,0,1000,v1-ashery-chat-split,mock_random_regular_k1_s3441225456,3441225456,completed,2026-07-14T08:03:27.542404+00:00,55,55,random_regular


## t_gate histogram (eligible runs)

In [3]:
eligible = state.runs[state.runs["eligibility"] == 1]
fig, ax = plt.subplots(figsize=(6, 4))
if not eligible.empty and eligible["t_gate"].notna().any():
    ax.hist(eligible["t_gate"].dropna(), bins=10)
ax.set_xlabel("t_gate (interactions to Phase 1 gate)")
ax.set_ylabel("count")
ax.set_title("t_gate distribution, eligible runs")
plt.close(fig)
fig


<Figure size 600x400 with 1 Axes>

## Full dominant-share distribution

In [4]:
progress = state.events[state.events["type"].isin(["phase1_progress", "phase2_progress"])] if not state.events.empty else pd.DataFrame()
shares = progress["dominant_share"].dropna() if "dominant_share" in progress.columns else pd.Series(dtype=float)

fig, ax = plt.subplots(figsize=(6, 4))
if not shares.empty:
    ax.hist(shares, bins=20, range=(0, 1))
ax.set_xlabel("dominant label share")
ax.set_ylabel("count")
ax.set_title("Dominant-share distribution across all progress events")
plt.close(fig)
fig


<Figure size 600x400 with 1 Axes>

## Eligibility by topology

In [5]:
if not state.runs.empty:
    finished = state.runs[state.runs["run_status"].isin(["completed", "censored"])]
    eligibility_table = finished.groupby("topology").agg(
        attempted=("run_id", "count"), eligible=("eligibility", "sum")
    )
    eligibility_table["rate"] = eligibility_table["eligible"] / eligibility_table["attempted"]
else:
    eligibility_table = pd.DataFrame(columns=["attempted", "eligible", "rate"])
eligibility_table


,attempted,eligible,rate
topology,,,
complete,2,2,1.0
lattice,2,2,1.0
random_regular,2,2,1.0


## Cost summary

In [6]:
cost_summary(state.events)


,model_id,prompt_tokens,completion_tokens,n_calls,cost_usd
0,mock,0.0,0.0,10408,0.0


In [7]:
if USE_MOCK_SCRATCH_BATCH:
    _scratch.cleanup()
